In [2]:
from handwriting_sample import HandwritingSample as hs

svc_sample = hs.from_svc(path="C://dev//dolphin_initial_testing//DOLPHIN//data-raw//LBD_CZ_002//HC-12#1//HC-12#1_w.cz.fnusa.1_1.svc")
print(svc_sample)



DEBUG:fsspec.local:open file: C://dev//dolphin_initial_testing//DOLPHIN//data-raw//LBD_CZ_002//HC-12#1//HC-12#1_w.cz.fnusa.1_1.svc
2025-10-26 19:27:16 - 272 - SVCFileReader - Old file-name format no additional meta data
2025-10-26 19:27:16 - 272 - SVCFileReader - Data has been loaded from an SVC file: C://dev//dolphin_initial_testing//DOLPHIN//data-raw//LBD_CZ_002//HC-12#1//HC-12#1_w.cz.fnusa.1_1.svc
<HandwritingSampleObject: 
DATA:
   x =          [239.91  239.91  239.91  ... 267.82  267.8   267.645], 
   y =          [141.41  141.41  141.41  ... 134.06  134.155 134.465], 
   time =       [0.0000e+00 7.0000e-03 1.5000e-02 ... 1.7539e+01 1.7547e+01 1.7554e+01], 
   pen_status = [ True  True  True ...  True  True  True], 
   azimuth =    [1390. 1390. 1400. ... 1420. 1420. 1420.], 
   tilt =       [560. 560. 560. ... 590. 590. 590.], 
   pressure =   [0.002933 0.025415 0.047898 ... 0.260997 0.246334 0.057674]> 


METADATA:
dict_items([('samples_count', 2336)])


In [3]:
from handwriting_features import HandwritingFeatures as hf

data_path = "C://dev//dolphin_initial_testing//DOLPHIN//data-raw//LBD_CZ_002//HC-12#1//HC-12#1_w.cz.fnusa.1_1.svc"

variables = ["y", "x", "time", "pen_status", "azimuth", "tilt", "pressure"]

fs = 133  # Sampling frequency in Hz

feature_data = hf.from_svc(data_path, variables)

# 1. Kinematic features
x_velocity = feature_data.velocity(axis="x", in_air=False, statistics=["mean", "std"])
y_velocity = feature_data.velocity(axis="y", in_air=False, statistics=["mean", "std"])

pressure = feature_data.pressure(statistics=["median", "std"])

print("X Velocity Features:", x_velocity)
print("Y Velocity Features:", y_velocity)
print("Pressure Features:", pressure)

DEBUG:fsspec.local:open file: C://dev//dolphin_initial_testing//DOLPHIN//data-raw//LBD_CZ_002//HC-12#1//HC-12#1_w.cz.fnusa.1_1.svc
2025-10-26 19:27:19 - 272 - SVCFileReader - Old file-name format no additional meta data
2025-10-26 19:27:19 - 272 - SVCFileReader - Data has been loaded from an SVC file: C://dev//dolphin_initial_testing//DOLPHIN//data-raw//LBD_CZ_002//HC-12#1//HC-12#1_w.cz.fnusa.1_1.svc
X Velocity Features: [23.58355521 15.80903653]
Y Velocity Features: [26.17234362 20.51260332]
Pressure Features: [0.173021   0.03585308]


In [19]:
import os
from pathlib import Path
import pandas as pd
from handwriting_features import HandwritingFeatures as hf

directory_path = "C://dev//dolphin_initial_testing//DOLPHIN//data-raw//LBD_CZ_002"

pathlist = Path(directory_path).rglob("*.svc")

variables = ["y", "x", "time", "pen_status", "azimuth", "tilt", "pressure"]
x_velocity, y_velocity, pressure= [],[],[]
pd_rows = []
fs = 133  # Sampling frequency in Hz
i = 0

def diagnosis_from_filename(filename):
    if filename.startswith("HC"):
        return 1
    elif filename.startswith("pre-LBD"):
        return 0
    else:
        return None  # or raise an exception if appropriate
for path in pathlist:

    diagnosis = diagnosis_from_filename(path.name)
    x_velocity_loop = feature_data.velocity(axis="x", in_air=False, statistics=["mean", "std"])
    y_velocity_loop = feature_data.velocity(axis="y", in_air=False, statistics=["mean", "std"])
    pressure_loop = feature_data.pressure(statistics=["mean", "std"])
    
    x_velocity.append(x_velocity_loop)
    y_velocity.append(y_velocity_loop)
    pressure.append(pressure_loop)

    pandas_row = {
        "file_name": path.name,
        "diagnosis": diagnosis,
        "x_velocity_mean" : x_velocity_loop[0],
        "x_velocity_std" : x_velocity_loop[1],
        "y_velocity_mean" : y_velocity_loop[0],
        "y_velocity_std" : y_velocity_loop[1],
        "pressure_mean" : pressure_loop[0],
        "pressure_std" : pressure_loop[1]
    }
    pd_rows.append(pandas_row)
df = pd.DataFrame([pandas_row])
i = i + 1

print("Processed files:", i)
print(f"File: {path.name}, Diagnosis: {diagnosis}")
print("X Velocity Features:", x_velocity_loop)
print("Y Velocity Features:", y_velocity_loop)
print("Pressure Features:", pressure_loop)







Processed files: 1
File: pre-LBD-99#1_w.cz.fnusa.9_1.svc, Diagnosis: 0
X Velocity Features: [23.58355521 15.80903653]
Y Velocity Features: [26.17234362 20.51260332]
Pressure Features: [0.17806461 0.03585308]


In [11]:
from pathlib import Path
import pandas as pd
from handwriting_features import HandwritingFeatures as HF

# --- Config ---
root = Path("C:/dev/dolphin_initial_testing/DOLPHIN/data-raw/LBD_CZ_002")
fs = 133  # Hz
variables = ["y", "x", "time", "pen_status", "azimuth", "tilt", "pressure"]

# --- Helpers ---
def diagnosis_from_path(p: Path):
    """
    Returns 1 for HC, 0 for pre-LBD, or None if not found.
    Checks filename stem and all parent folders.
    """
    names = [p.stem] + [par.name for par in p.parents]
    for name in names:
        if name.startswith("HC"):
            return 1
        if name.startswith("pre-LBD"):
            return 0
    return None

def flatten_feature_dict(prefix: str, d: dict):
    """Prefix and flatten a feature dict, e.g. {'mean': 0.1} -> {'x_velocity_mean': 0.1}"""
    return {f"{prefix}_{k}": v for k, v in (d or {}).items()}

# --- Main loop ---
rows = []
processed = 0
skipped = 0
errors = []

for path in root.rglob("*.svc"):
    diag = diagnosis_from_path(path)
    if diag is None:
        skipped += 1
        continue

    try:
        # Build the per-file feature object (adjust if your API uses a different constructor)
        feat = HF(path, variables=variables, fs=fs)

        # Compute features (dicts with keys like {"mean": ..., "std": ...})
        x_velocity = feat.velocity(axis="x", in_air=False, statistics=["mean", "std"])
        y_velocity = feat.velocity(axis="y", in_air=False, statistics=["mean", "std"])
        pressure   = feat.pressure(statistics=["mean", "std"])

        # Flatten into one row
        row = {
            "file_name": path.name,
            "relpath": str(path.relative_to(root)),
            "diagnosis": diag,  # 1=HC, 0=pre-LBD
            **flatten_feature_dict("x_velocity", x_velocity),
            **flatten_feature_dict("y_velocity", y_velocity),
            **flatten_feature_dict("pressure", pressure),
        }
        rows.append(row)
        processed += 1

    except Exception as e:
        errors.append((str(path), repr(e)))

# --- Build DataFrame ---
df = pd.DataFrame(rows)

print(f"Processed files: {processed}")
print(f"Skipped (no diagnosis in path): {skipped}")
print(f"Errors: {len(errors)}")
if errors:
    # Print first few errors for quick debugging
    for pth, err in errors[:5]:
        print("ERR:", pth, "->", err)

# Peek
print(df.head())

# Optional: save for ML
# df.to_csv("lbd_features.csv", index=False)


Processed files: 0
Skipped (no diagnosis in path): 0
Errors: 1911
ERR: C:\dev\dolphin_initial_testing\DOLPHIN\data-raw\LBD_CZ_002\HC-1#1\HC-1#1_w.cz.fnusa.10_1.svc -> AttributeError("'WindowsPath' object has no attribute 'compute_velocity'")
ERR: C:\dev\dolphin_initial_testing\DOLPHIN\data-raw\LBD_CZ_002\HC-1#1\HC-1#1_w.cz.fnusa.15_1.svc -> AttributeError("'WindowsPath' object has no attribute 'compute_velocity'")
ERR: C:\dev\dolphin_initial_testing\DOLPHIN\data-raw\LBD_CZ_002\HC-1#1\HC-1#1_w.cz.fnusa.16_1.svc -> AttributeError("'WindowsPath' object has no attribute 'compute_velocity'")
ERR: C:\dev\dolphin_initial_testing\DOLPHIN\data-raw\LBD_CZ_002\HC-1#1\HC-1#1_w.cz.fnusa.17_1.svc -> AttributeError("'WindowsPath' object has no attribute 'compute_velocity'")
ERR: C:\dev\dolphin_initial_testing\DOLPHIN\data-raw\LBD_CZ_002\HC-1#1\HC-1#1_w.cz.fnusa.18_1.svc -> AttributeError("'WindowsPath' object has no attribute 'compute_velocity'")
Empty DataFrame
Columns: []
Index: []
